# 💳 Enterprise Fraud Detection — Cost-Sensitive LightGBM Pipeline
### Table 1: Machine Learning & Explainable AI Engine (Hari's Pipeline)
---
This Google Colab notebook implements **Cost-Sensitive LightGBM** with **asymmetric financial penalties ($10 \times \text{FN} + 1 \times \text{FP}$)** and **TreeSHAP local explainability**.

### 📌 How to provide the dataset (Choose One):
* **Option 1 (Google Drive):** If your files are in Google Drive, run **Cell 1** to mount Drive.
* **Option 2 (Direct Upload):** Click the 📁 **Folder icon** on the left sidebar and drag-and-drop `train_transaction.csv` and `train_identity.csv` into `/content`.
* **Option 3 (Kaggle API):** Use the Kaggle API cell below to download automatically.
* **Option 4 (Instant Demo Mode):** If you run without uploading, the notebook **automatically generates a 10,000-row benchmark dataset** so you can test the full pipeline, see the charts, and export JSON immediately!

## 1. Setup & Dependencies
Mount Google Drive (optional) and install required dependencies.

In [ ]:
# Optional: Mount Google Drive if your datasets are in Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully!')
except Exception:
    print('Running locally or Drive already mounted.')

# Install packages
!pip install -q lightgbm shap scikit-learn matplotlib

### (Optional) Download IEEE-CIS Dataset via Kaggle API
Run this cell only if you want to download the 1.2 GB dataset directly from Kaggle. Otherwise, skip to Section 2.

In [ ]:
# To use Kaggle API: upload your kaggle.json file to /content, then uncomment and run:
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c ieee-fraud-detection
# !unzip -q -o train_transaction.csv.zip
# !unzip -q -o train_identity.csv.zip
# print('Dataset downloaded and unzipped!')

## 2. Dataset Discovery & Loading
Searches for dataset files across Drive, `/content`, or activates Instant Demo Mode if files are not present.

In [ ]:
import os, sys, gc, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, roc_curve

warnings.filterwarnings('ignore')
np.random.seed(42)

# Search common locations
def find_csv(filename):
    candidates = [
        Path('/content') / filename,
        Path('/content/drive/MyDrive') / filename,
        Path('/content/drive/MyDrive/ieee-fraud-detection') / filename,
        Path('/content/drive/MyDrive/IEEE-CIS') / filename,
        Path.cwd() / filename,
        Path.cwd() / 'data' / 'ieee-fraud-detection' / filename,
    ]
    for c in candidates:
        if c.is_file(): return c
    for p in Path('/content').rglob(filename):
        if p.is_file(): return p
    return None

txn_path = find_csv('train_transaction.csv')
id_path = find_csv('train_identity.csv')

use_demo_data = False
if txn_path and id_path:
    print(f'[FOUND] Real Transaction file: {txn_path}')
    print(f'[FOUND] Real Identity file:    {id_path}')
else:
    print('[INFO] Raw CSV files not found in /content or Drive.')
    print('[INFO] >> ACTIVATING INSTANT DEMO MODE: Generating 10,000-sample benchmark dataset...')
    use_demo_data = True

    n_demo = 10000
    demo_ids = np.arange(3000000, 3000000 + n_demo)
    demo_dt = np.sort(np.random.randint(86400, 86400 * 90, size=n_demo))
    demo_fraud = (np.random.rand(n_demo) < 0.035).astype(int)
    demo_amt = np.round(np.random.exponential(scale=65.0, size=n_demo) + 5.0, 2)
    # Fraud tends to have larger amounts
    demo_amt[demo_fraud == 1] *= np.random.uniform(2.0, 5.0, size=int(demo_fraud.sum()))
    demo_card1 = np.random.randint(1000, 19999, size=n_demo)
    demo_card2 = np.random.randint(100, 600, size=n_demo)
    demo_addr1 = np.random.randint(100, 500, size=n_demo)
    demo_p_email = np.random.choice(['gmail.com', 'yahoo.com', 'hotmail.com', 'anonymous.com', 'corp.com'], size=n_demo)
    demo_prod = np.random.choice(['W', 'C', 'R', 'H', 'S'], size=n_demo)

    demo_dict = {
        'TransactionID': demo_ids,
        'isFraud': demo_fraud,
        'TransactionDT': demo_dt,
        'TransactionAmt': demo_amt,
        'ProductCD': demo_prod,
        'card1': demo_card1,
        'card2': demo_card2,
        'addr1': demo_addr1,
        'P_emaildomain': demo_p_email,
    }
    for i in range(1, 15):
        demo_dict[f'C{i}'] = np.random.poisson(lam=1.5, size=n_demo)
    for i in range(1, 16):
        demo_dict[f'D{i}'] = np.random.randint(0, 300, size=n_demo).astype(float)
    for i in range(1, 10):
        demo_dict[f'M{i}'] = np.random.choice(['T', 'F', 'M0', 'M1', np.nan], size=n_demo)

    df = pd.DataFrame(demo_dict)
    print(f'[SUCCESS] Generated {len(df):,} benchmark transactions (Fraud rate: {df["isFraud"].mean()*100:.2f}%)')
    print('You can now run all remaining cells seamlessly!')

## 3. Data Preprocessing & Feature Engineering
Merges datasets (if real files found), creates temporal features (`TransactionDay`, `TransactionHour`), relative amount ratios, encodes categoricals, and sorts chronologically.

In [ ]:
if not use_demo_data:
    print('Loading and merging raw CSV datasets...')
    txn_df = pd.read_csv(txn_path)
    id_df = pd.read_csv(id_path)
    df = pd.merge(txn_df, id_df, on='TransactionID', how='left')
    del txn_df, id_df
    gc.collect()

    # Prune ultra-sparse V-columns
    v_drop = [f'V{i}' for i in range(35, 279) if f'V{i}' in df.columns]
    if v_drop: df.drop(columns=v_drop, inplace=True)

# Sort strictly chronologically
df.sort_values('TransactionDT', inplace=True)
df.reset_index(drop=True, inplace=True)

# Temporal & Behavioral Feature Engineering
df['TransactionDay'] = (df['TransactionDT'] // (24 * 3600)).astype(np.int16)
df['TransactionHour'] = ((df['TransactionDT'] // 3600) % 24).astype(np.int8)
if 'card1' in df.columns:
    c_mean = df.groupby('card1')['TransactionAmt'].transform('mean')
    df['TransactionAmt_to_mean_card1'] = (df['TransactionAmt'] / (c_mean + 1.0)).astype(np.float32)

# Frequency encoding
for c in ['card1', 'card2', 'addr1', 'P_emaildomain']:
    if c in df.columns:
        df[f'{c}_freq'] = df[c].map(df[c].value_counts(dropna=False)).astype(np.float32)

# Categorical integer encoding
cat_cols = df.select_dtypes(include=['object']).columns
for c in cat_cols:
    df[c] = df[c].astype('category').cat.codes.astype(np.int32)

# Downcast float64 to float32
for c in df.select_dtypes(include=['float64']).columns:
    df[c] = df[c].astype(np.float32)

print(f'[INFO] Final feature matrix: {df.shape[0]:,} rows x {df.shape[1]} columns | Fraud: {df["isFraud"].mean()*100:.2f}%')

## 4. Custom Cost-Sensitive Objective Function
Implements the asymmetric 10:1 penalty matrix ($C_{FN}=10.0, C_{FP}=1.0$) directly into LightGBM's gradient and hessian calculations.

In [ ]:
C_FP = 1.0
C_FN = 10.0

def cost_sensitive_objective(y_pred, dtrain):
    y_true = dtrain.get_label().astype(np.float64)
    p = 1.0 / (1.0 + np.exp(-np.clip(y_pred, -15.0, 15.0)))
    weights = np.where(y_true == 1, C_FN, C_FP)
    grad = weights * (p - y_true)
    hess = np.maximum(weights * p * (1.0 - p), 1e-8)
    return grad, hess

def cost_sensitive_eval(y_pred, dtrain):
    y_true = dtrain.get_label().astype(int)
    p = 1.0 / (1.0 + np.exp(-np.clip(y_pred, -15.0, 15.0)))
    y_hat = (p >= 0.5).astype(int)
    fp = int(np.sum((y_hat == 1) & (y_true == 0)))
    fn = int(np.sum((y_hat == 0) & (y_true == 1)))
    total_cost = C_FP * fp + C_FN * fn
    return 'total_cost', total_cost, False

## 5. Chronological Split & Booster Training
Trains on the past 80% and evaluates on the untouched latest 20% to avoid future lookahead bias.

In [ ]:
split = int(len(df) * 0.80)
train_df = df.iloc[:split]
test_df = df.iloc[split:]

features = [c for c in df.columns if c not in ['TransactionID', 'isFraud', 'TransactionDT']]
X_train, y_train = train_df[features], train_df['isFraud'].values
X_test, y_test = test_df[features], test_df['isFraud'].values
test_ids = test_df['TransactionID'].values

print(f'Train set: {len(X_train):,} samples | Test set: {len(X_test):,} samples')
dtrain = lgb.Dataset(X_train, label=y_train)
dtest = lgb.Dataset(X_test, label=y_test, reference=dtrain)

params = {
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'feature_fraction': 0.80,
    'bagging_fraction': 0.80,
    'bagging_freq': 1,
    'verbosity': -1,
    'seed': 42
}

print('Training Cost-Sensitive Booster (Asymmetric Gradient Optimization)...')
model = lgb.train(
    params,
    dtrain,
    num_boost_round=1200,
    valid_sets=[dtrain, dtest],
    valid_names=['train', 'valid'],
    fobj=cost_sensitive_objective,
    feval=cost_sensitive_eval,
    callbacks=[lgb.early_stopping(stopping_rounds=60, verbose=True)]
)

## 6. Financial Evaluation & Benchmark Curves

In [ ]:
raw_preds = model.predict(X_test, num_iteration=model.best_iteration)
y_prob = 1.0 / (1.0 + np.exp(-np.clip(raw_preds, -15.0, 15.0)))

auc = roc_auc_score(y_test, y_prob)
print('=' * 65)
print(f'Final Test ROC-AUC: {auc:.4f}')

# Evaluate at 0.50 vs Optimal Threshold
y_hat_05 = (y_prob >= 0.5).astype(int)
cm_05 = confusion_matrix(y_test, y_hat_05)
tn, fp, fn, tp = cm_05.ravel()
cost = C_FP * fp + C_FN * fn
print(f'Total Financial Cost at 0.50 threshold: {cost:,} (FP={fp:,}, FN={fn:,})')
print('=' * 65)

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'ROC-AUC = {auc:.4f}', color='#2563eb', lw=2)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Cost-Sensitive LightGBM — Final Test ROC')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. TreeSHAP Explainability & Handoff Export
Computes exact local Shapley values and exports `lightgbm_shap_output.json` for live Table 3 integration.

In [ ]:
os.makedirs('outputs', exist_ok=True)
n_explain = min(5000, len(X_test))
X_exp = X_test.iloc[:n_explain]
ids_exp = test_ids[:n_explain]
probs_exp = y_prob[:n_explain]

print(f'Computing TreeSHAP contributions for {n_explain:,} transactions...')
contribs = model.predict(X_exp, num_iteration=model.best_iteration, pred_contrib=True)
shap_mat = contribs[:, :-1]

records = []
for i in range(n_explain):
    top_idx = np.argsort(np.abs(shap_mat[i]))[-3:][::-1]
    records.append({
        'transaction_id': str(ids_exp[i]),
        'dataset_name': 'ieee_cis',
        'prediction_probability': round(float(probs_exp[i]), 6),
        'top3_shap_features': {features[j]: round(float(shap_mat[i, j]), 6) for j in top_idx}
    })

with open('outputs/lightgbm_shap_output.json', 'w') as f:
    json.dump(records, f, indent=2)
print('[SUCCESS] Exported outputs/lightgbm_shap_output.json')

# Summary plot
mean_shap = np.mean(np.abs(shap_mat), axis=0)
top20 = np.argsort(mean_shap)[-20:]
plt.figure(figsize=(10, 8))
plt.barh([features[j] for j in top20], mean_shap[top20], color='#2563eb')
plt.xlabel('Mean |TreeSHAP| Contribution')
plt.title('Global Feature Importance')
plt.tight_layout()
plt.savefig('outputs/shap_summary.png', dpi=160)
plt.show()
print('[SUCCESS] Saved outputs/shap_summary.png')

# Automatic Colab file download
try:
    from google.colab import files
    files.download('outputs/lightgbm_shap_output.json')
    files.download('outputs/shap_summary.png')
except Exception:
    print('Deliverables available in outputs/ directory.')